<h1>ReAct: Build Reasoning and Acting AI Agents with LangGraph</h1>

You're a software engineer on a mission: build an AI agent that doesn't just respond—it thinks. In this lab, you'll step into the role of an AI architect, designing a smart assistant that solves tough problems by reasoning through them and taking purposeful actions.

Using the ReAct (Reasoning + Acting) framework, you'll teach your agent to think step by step, consult tools like search engines or calculators, and adapt on the fly. It’s not just about answers—it’s about how the agent gets there.

By the end of the lab, your AI will <u>face a mystery that can’t be solved with knowledge alone. It will need logic, resourcefulness, and the ability to act—just like you</u>, the engineer who built it.


<h2>What is ReAct?</h2>

ReAct stands for Reasoning + Acting. It's a framework that combines:

- Reasoning: The agent thinks through problems step by step, maintaining an internal dialogue about what it needs to do.
- Acting: The agent can use external tools (search engines, calculators, APIs) to gather information or perform actions.
- Observing: The agent processes the results from its actions and incorporates them into its reasoning.

This creates a powerful loop: Think → Act → Observe → Think → Act → ...


<h2>Why ReAct Matters</h2>
Traditional language models are limited by their training data cutoff and can't access real-time information. ReAct agents overcome tradition LLM limitation by:

- Accessing current information through web searches
- Performing calculations with specialized tools
- Breaking down complex problems into manageable steps
- Adapting their approach based on intermediate results


<h2>Objectives</h2>
After completing this lab you will be able to:

- Use the ReAct framework to solve multi-step problems with external tools
- Teach an AI agent to reason step by step, take actions, and adapt based on results
- Build a smart assistant that can handle tasks requiring logic and tool use


<h2>Setup & Installation</h2>
For this lab, we will be using the following libraries:

- LangGraph: A framework for building stateful, multi-step AI applications using graphs.
- LangChain: A toolkit that provides tools and abstractions for working with language models.
- LangChain-Ollama: Ollama integration for LangChain.
- LangChain-Community: Community-contributed tools and integrations.

<p>
Use Python 3.12.10

In [12]:
%%capture
%pip install langgraph==0.3.34 
%pip install langchain-openai==0.3.14 
%pip install langchainhub==0.1.21 
%pip install langchain==0.3.24 
%pip install pygraphviz==1.14 
%pip install langchain-community==0.3.23
%pip install langchain-ollama==0.3.10

In [13]:
import os
import json
import getpass
from typing import List, Dict
from langchain_community.utilities.tavily_search import TavilySearchAPIWrapper
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import END, StateGraph, MessagesState

In [4]:
# copy your Tavily API key in the pop up shown when executing this cell

def _set_if_undefined(var: str) -> None:
    if os.environ.get(var):
      return
    os.environ[var] = getpass.getpass(var)
_set_if_undefined("TAVILY_API_KEY")

In [14]:
import warnings 
warnings.filterwarnings('ignore')

from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.tools import tool
import os
import json


# Initialize the Tavily search tool
search = TavilySearchResults()

@tool
def search_tool(query: str):
    """
    Search the web for information using Tavily API.

    :param query: The search query string
    :return: Search results related to the query
    """
    return search.invoke(query)

In [6]:
search_tool.invoke("What's the weather like in Tokyo today?")

[{'title': 'Japan Weather in October 2026: Foliage & What to Wear',
  'url': 'https://www.umetravel.com/japan-weather/weather-in-october.html',
  'content': "Hokkaido has the earliest autumn foliage in Japan\n\nHokkaido has the earliest autumn foliage in Japan\n\n### Central Japan: Warm Days, Cool Nights—Perfect for Sightseeing\n\nTokyo· Yokohama· Kyoto· Osaka· Nara· Nagoya\n\nWeather features: Comfortable warmth during the day, cooler evenings, low rainfall\n\n|  |  |  |  |\n ---  --- |\n| City | Avg. High | Avg. Low | Rain Chance |\n| Tokyo | 22°C (72°F) | 15°C (59°F) | 27% – 32% |\n| Osaka | 23°C (73°F) | 15°C (59°F) | 25% – 31% |\n| Kyoto | 22°C (72°F) | 13°C (55°F) | 28% – 34% |\n| Nagoya | 23°C (73°F) | 14°C (57°F) | 26% – 33% |\n\nComfortable climate: Central Japan enjoys some of the most pleasant weather in October. Days remain mild to warm, especially early in the month, while evenings turn cool and refreshing. [...] UME Travel logo\nUME Travel Mobile logo\n\n# Japan Weather i

In [7]:
@tool
def recommend_clothing(weather: str) -> str:
    """
    Returns a clothing recommendation based on the provided weather description.

    This function examines the input string for specific keywords or temperature indicators 
    (e.g., "snow", "freezing", "rain", "85°F") to suggest appropriate attire. It handles 
    common weather conditions like snow, rain, heat, and cold by providing simple and practical 
    clothing advice.

    :param weather: A brief description of the weather (e.g., "Overcast, 64.9°F")
    :return: A string with clothing recommendations suitable for the weather
    """
    weather = weather.lower()
    if "snow" in weather or "freezing" in weather:
        return "Wear a heavy coat, gloves, and boots."
    elif "rain" in weather or "wet" in weather:
        return "Bring a raincoat and waterproof shoes."
    elif "hot" in weather or "85" in weather:
        return "T-shirt, shorts, and sunscreen recommended."
    elif "cold" in weather or "50" in weather:
        return "Wear a warm jacket or sweater."
    else:
        return "A light jacket should be fine."

In [8]:
tools=[search_tool,recommend_clothing]

tools_by_name={ tool.name:tool for tool in tools}

In [15]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("gemma4", model_provider="ollama")

In [16]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage,SystemMessage

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a helpful AI assistant that thinks step-by-step and uses tools when needed.

When responding to queries:
1. First, think about what information you need
2. Use available tools if you need current data or specific capabilities  
3. Provide clear, helpful responses based on your reasoning and any tool results

Always explain your thinking process to help users understand your approach.
"""),
    MessagesPlaceholder(variable_name="scratch_pad")
])

In [17]:
llm_with_tools = llm.bind_tools(tools)
model_react=chat_prompt| llm_with_tools

In [18]:
from typing import (Annotated,Sequence,TypedDict)
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages

# In ReAct, state management is crucial, as the agent must maintain context across multiple reasoning and acting steps.
class AgentState(TypedDict):
    """The state of the agent."""

    # add_messages is a reducer
    # See https://langchain-ai.github.io/langgraph/concepts/low_level/#reducers
    messages: Annotated[Sequence[BaseMessage], add_messages]

In [19]:
# Example conversation flow:
state: AgentState = {"messages": []}

# append a message using the reducer properly
state["messages"] = add_messages(state["messages"], [HumanMessage(content="Hi")])
print("After greeting:", state["messages"])

# add another message (e.g. a question)
state["messages"] = add_messages(state["messages"], [HumanMessage(content="Weather in NYC?")])
print("After question:", state)

After greeting: [HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}, id='51e6d2fe-28db-4e53-96f5-a39542806350')]
After question: {'messages': [HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}, id='51e6d2fe-28db-4e53-96f5-a39542806350'), HumanMessage(content='Weather in NYC?', additional_kwargs={}, response_metadata={}, id='b6133eaa-9b39-4490-9a98-a0800c1e7439')]}


In [ ]:
# Manual ReAct Execution (Understanding the Flow)


#Step1 : query processing

dummy_state: AgentState = {
    "messages": [HumanMessage( "What's the weather like in Zurich, and what should I wear based on the temperature?")]}

response = model_react.invoke({"scratch_pad":dummy_state["messages"]})

dummy_state["messages"]=add_messages(dummy_state["messages"],[response])

# step2 : tool execution
tool_call = response.tool_calls[-1]
print("Tool call:", tool_call)

tool_result = tools_by_name[tool_call["name"]].invoke(tool_call["args"])
print("Tool result preview:", tool_result[0]['title'])

tool_message = ToolMessage(
    content=json.dumps(tool_result),
    name=tool_call["name"],
    tool_call_id=tool_call["id"]
)
dummy_state["messages"] = add_messages(dummy_state["messages"], [tool_message])

# Step 3: Processing Results and Next Action

response = model_react.invoke({"scratch_pad": dummy_state["messages"]})
dummy_state['messages'] = add_messages(dummy_state['messages'], [response])

# check if the model wants to use another tool
if response.tool_calls:
    tool_call = response.tool_calls[0]
    tool_result = tools_by_name[tool_call["name"]].invoke(tool_call["args"])
    tool_message = ToolMessage(
        content=json.dumps(tool_result),
        name=tool_call["name"],
        tool_call_id=tool_call["id"]
    )
dummy_state['messages'] = add_messages(dummy_state['messages'], [tool_message])

# Step 4: Final Response Generation
response = model_react.invoke({"scratch_pad": dummy_state["messages"]})
print("Final response generated:", response.content is not None)
print("More tools needed:", bool(response.tool_calls))


# What Happens Here: 1. The model has all necessary information. 2. It synthesizes weather data and clothing recommendations.
# 3. It generates a comprehensive response to the user. 4. No more tool calls needed—the reasoning cycle is complete.


Tool call: {'name': 'search_tool', 'args': {'query': 'weather in Zurich'}, 'id': '5d513782-e9a7-42f0-82b6-f0215d32ef28', 'type': 'tool_call'}
Tool result preview: Zurich Weather Conditions: Temperature | 30 Days Forecast
Final response generated: True
More tools needed: False
